# 파일럿 — DeepAries(CIKM 2025) 공식 코드 실행 가능성

**목적(사전 고정).** 재현·재평가 연구의 대상으로 쓸 수 있는지 **실행 가능성만** 확인한다.
1. 공식 코드(`dmis-lab/DeepAries` @ `634f559`)가 무료 Colab에서 수정 없이 도는가
2. 학습 1회(시드 1개)에 걸리는 시간 → 논문 절차(시드 10개 중 검증 손실 최소 1개 보고)의 총비용 추정
3. 코드 기본 설정의 테스트 성과를 논문 Table 1 DJ30(CAGR 7.90%, MDD 13.67%)과 나란히 기록 — **판정 아님**, 기술 기록

**이 노트북에서 하지 않는 것.** 등가중·buy-and-hold 비교, 비용률 변경, 시작 상태 변경, 행동 주입 점검(C1·S1). 이것들은 사전 등록 후 별도 노트북에서 한다. 정적 예측은 저장소 `notes/deeparies_static_reading.md`에 먼저 커밋했다.

**런타임.** CPU·GPU 어느 쪽이든 된다(코드가 자동 감지). 어느 쪽에서 돌렸는지 결과에 기록된다. 결과는 Drive `data/deeparies_pilot/`에 저장된다. 연결이 끊기면 "모두 실행"으로 다시 돌리면 끝난 단계는 건너뛴다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
OUT = pathlib.Path('/content/drive/MyDrive/data/deeparies_pilot'); OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# 공식 코드를 고정 커밋으로 받는다(데이터·체크포인트 포함, 약 110MB)
COMMIT = '634f559cf4603eeb2ba7509e31319d076b21c16c'
!test -d /content/DeepAries || git clone -q https://github.com/dmis-lab/DeepAries.git /content/DeepAries
!cd /content/DeepAries && git checkout -q $COMMIT && git log -1 --format='%H %cd'
# requirements.txt의 고정 버전 대신 Colab 기본 numpy/pandas/torch를 쓰고 없는 패키지만 설치한다(버전은 아래에 기록)
!pip install -q yfinance timm einops

In [ ]:
import sys, platform, subprocess, json, time, re, os
import numpy, pandas, scipy, torch
ENV = dict(python=sys.version.split()[0], numpy=numpy.__version__, pandas=pandas.__version__, scipy=scipy.__version__,
           torch=torch.__version__, cuda=torch.cuda.is_available(),
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, cpus=os.cpu_count())
print(ENV); (OUT / 'env.json').write_text(json.dumps(ENV, indent=1))

PAT = {'final_pv': r'Final Portfolio Value: ([-\d.]+)', 'total_ret': r'Total Return: ([-\d.]+)%',
       'cagr': r'Annualized Return: ([-\d.]+)%', 'mdd': r'Maximum Drawdown: ([-\d.]+)%'}
def run(tag, args):
    log = OUT / f'{tag}.log'
    if log.exists() and 'Maximum Drawdown' in log.read_text():
        txt = log.read_text(); sec = float(re.search(r'__ELAPSED__ ([\d.]+)', txt).group(1)); print(tag, '이미 완료 — 건너뜀')
    else:
        t0 = time.time()
        p = subprocess.run(['python', 'main.py'] + args, cwd='/content/DeepAries', capture_output=True, text=True)
        sec = time.time() - t0
        txt = p.stdout + '\n' + p.stderr + f'\n__ELAPSED__ {sec:.1f}\n'
        log.write_text(txt)
        if p.returncode != 0:
            print(txt[-3000:]); raise RuntimeError(f'{tag} 실패(returncode {p.returncode})')
    r = {k: (float(m.group(1)) if (m := re.search(v, txt)) else None) for k, v in PAT.items()}
    r.update(tag=tag, minutes=round(sec / 60, 2)); print(r); return r

## P1. 공개 체크포인트로 추론만
체크포인트의 분할(학습 < 2023-10-01, 테스트 ≥ 2024-01-01)은 논문 Table 1과 다르므로 논문 수치와 비교하지 않는다. 코드가 도는지만 본다.

In [ ]:
%%time
R = {}
R['P1'] = run('P1_pretrained_infer', ['--market', 'dj30', '--data', 'general', '--is_training', '0', '--seed', '0',
    '--valid_year', '2023-10-01', '--test_year', '2024-01-01', '--horizons', '1', '5', '20',
    '--checkpoint_dir', 'iTransformer_DeepAries_dj30_general_num_stocks(20)_sl(20)_pl(20)_pretrained'])

## P2. 코드 기본 설정으로 학습 1회(시드 0)
기본값: 학습 < 2020-12-31, 검증 ~2021-12-31, 테스트 ≥ 2021-12-31, `train_epochs=1`, fee 0.0001, iTransformer, 20종목. 논문에 분할·epoch 기재가 없어 코드 기본값을 공식 설정으로 본다.

In [ ]:
%%time
R['P2'] = run('P2_train_seed0_default', ['--market', 'dj30', '--data', 'general', '--is_training', '1', '--seed', '0'])

In [ ]:
# 요약 — 논문 Table 1 DJ30 DeepAries: CAGR 7.90%, MDD 13.67% (판정 아님, 나란히 기록)
import pandas as pd
df = pd.DataFrame(R.values()).set_index('tag')
df.loc['paper_table1_dj30'] = {'cagr': 7.90, 'mdd': -13.67}
df.to_csv(OUT / 'pilot_summary.csv'); print(df)
m = R['P2']['minutes']
print(f'\n학습 1회 {m:.1f}분 → 논문 절차(시드 10) 약 {10*m/60:.1f}시간, 조건 3개(무비용·비용평가·비용학습) × 시드 10 약 {20*m/60:.1f}시간(비용평가는 재학습 없음)')
print('\n이 셀 출력 전체를 복사해 Claude에게 붙여 넣어 주세요.')